In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
!pip install pyannote.audio faster_whisper pysrt  audio-separator[gpu]  ffmpeg-downloader
!python -c "import torch; torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True"

In [ ]:
import torch
import os
# ====================== 配置 ======================
config = {"path": {
    "pre": "0pre",
    "audio": "1audio",
    "vad": "2vad",
    "asr": "3asr",
    "model": "/content/model",
    "ffmpeg": "ffmpeg",
}, "model": {
    "sep": "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
    "vad": "4evergr8/pyannote-segmentation-3.0",
    "asr": "4evergr8/whisper-large-v2-translate-zh-v0.2-st-ct2",
}, "vad": {
    "min_duration_on": 0.3,
    "min_duration_off": 0.1,
    "sample_rate": 16000,
}, "sep": {
    "sample_rate": 44100,
    "chunk_duration":1800
# ====================== 设备与精度动态计算 ======================
}, "device": "cuda" if torch.cuda.is_available() else "cpu"}
config["compute_type"] = "float16" if config["device"] == "cuda" else "int8"
workpath="/content/gdrive/MyDrive/ASMRASR"
os.chdir(workpath)

In [ ]:
import ffmpeg_downloader
import os
from audio_separator.separator import Separator
import subprocess
import sys
subprocess.run(
    [sys.executable, "-m", "ffmpeg_downloader", "install", "8.0@full-shared"],
    input="y\n",
    text=True
)
ffmpeg_downloader.add_path()
if not os.path.exists(config["path"]["pre"]) or not os.listdir(config["path"]["pre"]):
    print(f"错误：目录不存在或为空")
    exit(0)
separator = Separator(
    model_file_dir=config["path"]["model"],
    output_dir=config["path"]["audio"],  # 分离后的音频存放在 1audio
    output_single_stem="vocals",  # 仅提取人声
    sample_rate=config["sep"]["sample_rate"],  # 44100
    use_autocast=True,
    chunk_duration=config["sep"]["chunk_duration"],
    mdxc_params={
        "segment_size": 256,
        "override_model_segment_size": True,
        "batch_size": 8,
        "overlap": 8,
        "pitch_shift": 0
    },
)
print(f"正在加载模型: {config['model']['sep']}")
separator.load_model(model_filename=config["model"]["sep"])
print(f"开始批量处理目录 ...")
separator.separate(config["path"]["pre"])
print("\n所有文件人声分离处理完成！")

In [ ]:
import os
import glob
import subprocess
import sys
from pathlib import Path
import ffmpeg_downloader
import torch
import pysrt
subprocess.run(
    [sys.executable, "-m", "ffmpeg_downloader", "install", "8.0@full-shared"],
    input="y\n",
    text=True
)
ffmpeg_downloader.add_path()
from pyannote.audio import Model
from pyannote.audio.pipelines import VoiceActivityDetection
extensions = ("wav", "mp3", "flac")
pattern = os.path.join(config["path"]["audio"], "*.*")
audio_files = [
    f for f in glob.glob(pattern)
    if f.lower().endswith(extensions)
]

if not audio_files:
    print(f"在 {config['path']['audio']} 中没有找到可处理的音频文件")
    exit(0)

print('设备:', config["device"])

# 2. 初始化 VAD 模型
vad = Model.from_pretrained(
    checkpoint=config["model"]["vad"],
    cache_dir=config["path"]["model"]
)
vad.to(torch.device(config["device"]))

vad_pipeline = VoiceActivityDetection(segmentation=vad)
# 注意：修正了之前代码中的变量名错误 config.vad_min.duration_off -> config["vad"]["min_duration_off"]
vad_pipeline.instantiate({
    "min_duration_on": config["vad"]["min_duration_on"],
    "min_duration_off": config["vad"]["min_duration_off"],
})

# 3. 开始循环处理
for audio_path in audio_files:
    print(f"\n处理音频: {audio_path}")

    file_obj = Path(audio_path)
    basename = file_obj.stem
    vad_log_path = os.path.join(config["path"]["vad"], f"{basename}.srt")



    # 4. 执行 VAD 识别
    vad_result = vad_pipeline(audio_path)

    # 5. 生成 SRT
    srt = pysrt.SubRipFile()
    for idx, (segment, _, _) in enumerate(vad_result.itertracks(yield_label=True), start=1):
        sub_item = pysrt.SubRipItem(
            index=idx,
            start=pysrt.SubRipTime.from_ordinal(int(segment.start * 1000)),
            end=pysrt.SubRipTime.from_ordinal(int(segment.end * 1000)),
            text=f"Speech_{idx}"
        )
        srt.append(sub_item)

    srt.save(vad_log_path, encoding='utf-8')
    print(f"VAD记录写入完成: {vad_log_path}")

print("\n所有音频 VAD 处理完毕！")

In [ ]:
import os
import glob
from pathlib import Path
import pysrt
import faster_whisper.vad
import faster_whisper.transcribe
from unittest.mock import patch
from faster_whisper import WhisperModel

# 1. 获取任务列表
extensions = ("wav", "mp3", "aac")
pattern = os.path.join(config["path"]["audio"], "*.*")
files = [
    f for f in glob.glob(pattern)
    if f.lower().endswith(extensions)
]

if not files:
    # 修正：使用字典访问 path["audio"]
    print(f"在 {config['path']['audio']} 中未找到音频文件")
    exit(0)

audio_tasks = []
for file_str in files:
    file_path = Path(file_str)
    basename = file_path.stem
    # 构造 VAD 输入和 ASR 输出路径
    vad_log_path = os.path.join(config["path"]["vad"], f"{basename}.srt")
    asr_log_path = os.path.join(config["path"]["asr"], f"{basename}.srt")

    if not os.path.exists(vad_log_path):
        print(f"跳过：找不到 VAD 字幕文件 {vad_log_path}")
        continue
    audio_tasks.append((str(file_path), vad_log_path, asr_log_path))

print('设备:', config["device"], '类型:', config["compute_type"])


# 2. 定义补丁函数（用于注入你之前 VAD 得到的时间轴）
def get_custom_vad_patch(audio, vad_parameters=None):
    if hasattr(faster_whisper.vad, "_custom_segments"):
        segment_count = len(faster_whisper.vad._custom_segments)
        print(f"  [Patch Success] 注入来自 SRT 的 {segment_count} 个片段")
        return faster_whisper.vad._custom_segments
    print("  [Patch Warning] 未发现预设片段数据！")
    return []


# 3. 初始化 Whisper 模型
model = WhisperModel(
    model_size_or_path=config["model"]["asr"],
    device=config["device"],
    compute_type=config["compute_type"],
    download_root=config["path"]["model"]
)

# 4. 开始推理循环
for audio_path, vad_log_path, asr_log_path in audio_tasks:
    print(f"\n开始推理: {Path(audio_path).name}")

    # 加载 VAD 得到的片段
    vad_subs = pysrt.open(vad_log_path)
    custom_chunks = []
    sr = config["vad"]["sample_rate"]  # 16000

    for sub in vad_subs:
        custom_chunks.append({
            'start': int(sub.start.ordinal / 1000 * sr),
            'end': int(sub.end.ordinal / 1000 * sr)
        })

    # 临时挂载到 faster_whisper 模块上
    faster_whisper.vad._custom_segments = custom_chunks

    # 使用 patch 强制 Whisper 使用我们的时间轴，而不是让它重新检测 VAD
    with patch(
            "faster_whisper.transcribe.get_speech_timestamps",
            side_effect=get_custom_vad_patch
    ):
        segments, _ = model.transcribe(
            audio=audio_path,
            language="ja",
            task='translate',
            vad_filter=True,  # 必须开启，Patch 才会生效
            condition_on_previous_text=True
        )

        asr_subs = pysrt.SubRipFile()
        for segment in segments:
            text = segment.text.strip()
            if not text: continue

            new_sub = pysrt.SubRipItem(
                index=len(asr_subs) + 1,
                start=pysrt.SubRipTime.from_ordinal(int(segment.start * 1000)),
                end=pysrt.SubRipTime.from_ordinal(int(segment.end * 1000)),
                text=text
            )
            asr_subs.append(new_sub)
            print(f"[{segment.start:.2f}s -> {segment.end:.2f}s]: {text}")

        # 处理完一个文件再保存，效率更高
        asr_subs.save(asr_log_path, encoding='utf-8')

    print(f"处理完成，保存至: {asr_log_path}")

print("\n所有 ASR 任务处理完毕！")